# ML Assignment 2

Name: Ravish H S  
ID: 2026AK05117  

Dataset used: Breast Cancer Wisconsin (UCI)  
I will train 5 classifiers, check all metrics, and save models for streamlit.

In [ ]:
# install stuff if missing on bits lab
import subprocess, sys
pkgs = ['scikit-learn', 'joblib', 'pandas', 'numpy', 'matplotlib', 'seaborn']
subprocess.run([sys.executable, '-m', 'pip', 'install', *pkgs, '--quiet'], check=True)
print('ok')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report
)
import joblib

SEED = 42
print('imports done')

## Load dataset

Need at least 12 features and 500 rows as per assignment.

In [ ]:
# same data as UCI breast cancer wisconsin
raw = load_breast_cancer(as_frame=True)
df = raw.frame.copy()
df = df.rename(columns={'target': 'diagnosis'})

print('shape:', df.shape)
print('features:', df.shape[1] - 1)
print('rows:', df.shape[0])
print('features ok (>=12)?', (df.shape[1] - 1) >= 12)
print('rows ok (>=500)?', df.shape[0] >= 500)
print('missing:', df.isnull().sum().sum())
print('class counts (0=malignant, 1=benign):')
print(df['diagnosis'].value_counts())
df.head()

## Train test split

In [ ]:
feat_cols = [c for c in df.columns if c != 'diagnosis']
X = df[feat_cols]
y = df['diagnosis']

# stratified so both classes stay in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

print('train:', X_train.shape[0])
print('test :', X_test.shape[0])
print('features:', X_train.shape[1])

## Define 5 models

1. Logistic Regression  
2. Decision Tree  
3. KNN  
4. Naive Bayes  
5. SVM  

I put StandardScaler inside Pipeline so scaling is saved with the model.

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, random_state=SEED))
    ]),
    'Decision Tree': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', DecisionTreeClassifier(random_state=SEED, max_depth=5))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5))
    ]),
    'Naive Bayes': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GaussianNB())
    ]),
    'Support Vector Machine': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', probability=True, random_state=SEED))
    ]),
}

print(list(models.keys()))

## Train and calculate metrics

Metrics needed: Accuracy, AUC, Precision, Recall, F1, MCC

In [ ]:
def calc(model, X_te, y_te):
    pred = model.predict(X_te)
    prob = model.predict_proba(X_te)[:, 1]
    return {
        'Accuracy': round(float(accuracy_score(y_te, pred)), 4),
        'AUC': round(float(roc_auc_score(y_te, prob)), 4),
        'Precision': round(float(precision_score(y_te, pred)), 4),
        'Recall': round(float(recall_score(y_te, pred)), 4),
        'F1': round(float(f1_score(y_te, pred)), 4),
        'MCC': round(float(matthews_corrcoef(y_te, pred)), 4),
    }, pred

results = []
preds = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    scores, pred = calc(model, X_test, y_test)
    preds[name] = pred
    row = {'ML Model Name': name}
    row.update(scores)
    results.append(row)
    print(name, scores)

results_df = pd.DataFrame(results)
results_df

## Comparison table

In [ ]:
print(results_df.to_string(index=False))
best_i = results_df['Accuracy'].idxmax()
print()
print('Best model:', results_df.loc[best_i, 'ML Model Name'])
print('Best accuracy:', results_df.loc[best_i, 'Accuracy'])

## Confusion matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for i, name in enumerate(models.keys()):
    cm = confusion_matrix(y_test, preds[name])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Mal', 'Ben'], yticklabels=['Mal', 'Ben'])
    axes[i].set_title(name)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

axes[-1].axis('off')
plt.tight_layout()
plt.show()

## My observations

- Logistic Regression gave the best overall numbers (accuracy, auc, f1, mcc)
- Decision Tree was the weakest among the five
- KNN recall was 1.0 on this split which is good, accuracy also high
- Naive Bayes auc is ok but accuracy/mcc are lower
- SVM is very close to Logistic Regression

**Winner for my dataset: Logistic Regression**

## Save models and test_data.csv

These are needed for github and streamlit app.

In [ ]:
os.makedirs('model', exist_ok=True)

# test csv must have features + target
test_out = X_test.copy()
test_out['diagnosis'] = y_test.values
test_out.to_csv('test_data.csv', index=False)
print('saved test_data.csv', len(test_out))

train_out = X_train.copy()
train_out['diagnosis'] = y_train.values
train_out.to_csv('train_data.csv', index=False)
df.to_csv('full_dataset.csv', index=False)

files = {
    'Logistic Regression': 'logistic_regression.joblib',
    'Decision Tree': 'decision_tree.joblib',
    'KNN': 'knn.joblib',
    'Naive Bayes': 'naive_bayes.joblib',
    'Support Vector Machine': 'svm.joblib',
}

for name, model in models.items():
    path = os.path.join('model', files[name])
    joblib.dump(model, path)
    print('saved', path)

meta = {
    'dataset': 'Breast Cancer Wisconsin (Diagnostic) - UCI',
    'source_url': 'https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic',
    'target_column': 'diagnosis',
    'target_meaning': {'0': 'malignant', '1': 'benign'},
    'feature_columns': feat_cols,
    'model_files': files,
    'metrics': [
        {
            'model': r['ML Model Name'],
            'accuracy': r['Accuracy'],
            'auc': r['AUC'],
            'precision': r['Precision'],
            'recall': r['Recall'],
            'f1': r['F1'],
            'mcc': r['MCC'],
        }
        for r in results
    ],
    'train_size': int(len(X_train)),
    'test_size': int(len(X_test)),
    'n_features': len(feat_cols),
    'n_instances': int(len(df)),
    'random_state': SEED,
}

with open(os.path.join('model', 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

print('meta.json saved')

In [ ]:
# just checking saved model works
m = joblib.load('model/logistic_regression.joblib')
scores, _ = calc(m, X_test, y_test)
print('reloaded LR:', scores)
print()
print(classification_report(y_test, preds['Logistic Regression'],
                            target_names=['Malignant', 'Benign']))
print('BITS Lab run finished')